# Lesson 01 — Convolution and Kernels: The Core Concept

## Why This Lesson
Every filter in image processing — blur, sharpen, edge detect — is a convolution.
A CNN's learned filters are convolutions. Understanding this once means understanding everything.

## What Convolution Does
Slide a small matrix (kernel) over the image. At each position, multiply each kernel value
by the overlapping pixel, sum them up — that's the output pixel.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Manual convolution from scratch (educational — never use this in production)
def manual_convolve(image, kernel):
    h, w   = image.shape
    kh, kw = kernel.shape
    pad_h, pad_w = kh//2, kw//2
    padded = np.pad(image.astype(np.float32), ((pad_h,pad_h),(pad_w,pad_w)), mode='reflect')
    output = np.zeros_like(image, dtype=np.float32)
    for i in range(h):
        for j in range(w):
            region        = padded[i:i+kh, j:j+kw]
            output[i, j]  = np.sum(region * kernel)
    return np.clip(output, 0, 255).astype(np.uint8)

img  = cv2.imread('sample.jpg')
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# Box blur kernel (average)
k_box = np.ones((5,5), dtype=np.float32) / 25.0

# Apply manually and with OpenCV — should be identical
manual_result = manual_convolve(gray, k_box)
opencv_result = cv2.filter2D(gray, -1, k_box)

print("Manual vs OpenCV max difference:", np.abs(manual_result.astype(int) - opencv_result.astype(int)).max())

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, im, t in zip(axes, [gray, manual_result, opencv_result],
    ['Original', 'Manual convolution (slow)', 'cv2.filter2D (fast, same result)']):
    ax.imshow(im, cmap='gray'); ax.set_title(t); ax.axis('off')
plt.show()

# Visualize different kernels and their effects
kernels = {
    'Identity':  np.array([[0,0,0],[0,1,0],[0,0,0]], dtype=np.float32),
    'Blur':      np.ones((5,5), dtype=np.float32)/25,
    'Sharpen':   np.array([[0,-1,0],[-1,5,-1],[0,-1,0]], dtype=np.float32),
    'Edge X':    np.array([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=np.float32),
}
fig, axes = plt.subplots(1, 4, figsize=(22,5))
for ax, (name, k) in zip(axes, kernels.items()):
    result = cv2.filter2D(gray, -1, k)
    ax.imshow(result, cmap='gray'); ax.set_title(name); ax.axis('off')
plt.suptitle('Same image, different kernels — convolution is the universal tool', fontsize=12)
plt.show()

## Key Takeaway
Convolution = slide kernel over image, multiply + sum at each position.
`cv2.filter2D(img, -1, kernel)` applies any kernel. -1 means output depth = input depth.